# ADS 505 Technical Notebook: Predicting U.S. Flight Departure Delays

**Team:** 4 (individual project, instructor-approved)  
**Author:** Michael Rosalia  
**GitHub repository:** https://github.com/mrosalia96/ads505-flight-delay-prediction

Predict pre-departure delay risk (≥ 15 minutes) for flights originating at SAN, LAX, PHX, DEN, and SEA in 2023, using BTS on-time performance and NOAA weather. Sections are added as the analysis progresses.

In [ ]:
# All package imports live in this first code cell.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BTS_DIR = RAW_DIR / "bts"
NOAA_DIR = RAW_DIR / "noaa"

ORIGIN_AIRPORTS = ("SAN", "LAX", "PHX", "DEN", "SEA")
ANALYSIS_YEAR = 2023

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from build_flights_sample import build_sample  # noqa: E402

print("Project root:", PROJECT_ROOT)

## 1. Problem statement and data description

**Problem.** Airlines and airports often learn about departure delays as they unfold. This project predicts **pre-departure delay risk** (15+ minutes late) so operations leaders can prioritize staffing, buffers, and passenger communication before pushback.

**Data.** U.S. DOT BTS Reporting Carrier On-Time Performance for calendar year 2023, filtered to origin airports SAN, LAX, PHX, DEN, and SEA. NOAA weather will be joined in a later step. Acquisition details and APA citations: `docs/data_sources.md`.

**Target.** Prefer BTS `DepDel15` (1 = departure delay ≥ 15 minutes). Rows with missing delay outcome or cancelled flights are handled explicitly below.

In [ ]:
# Build (or rebuild) the filtered origin sample from monthly BTS ZIPs.
# Requires: python src/download_bts.py --year 2023  (raw ZIPs in data/raw/bts/)
sample_path = build_sample(year=ANALYSIS_YEAR, origins=ORIGIN_AIRPORTS)
print("Sample file:", sample_path)

In [ ]:
# Load the processed sample and report basic shape / delay rate.
flights = pd.read_parquet(sample_path) if sample_path.suffix == ".parquet" else pd.read_csv(
    sample_path, parse_dates=["FlightDate"]
)

# Modelling set: exclude cancelled flights; require a defined 15-min delay flag.
mask_usable = (flights["Cancelled"] == 0) & flights["DepDel15"].notna()
flights_model = flights.loc[mask_usable].copy()
flights_model["DepDel15"] = flights_model["DepDel15"].astype(int)

print(f"Raw filtered rows: {len(flights):,}")
print(f"Usable (not cancelled, DepDel15 known): {len(flights_model):,}")
print(f"Date range: {flights_model['FlightDate'].min().date()} → {flights_model['FlightDate'].max().date()}")
print("Rows by origin:")
print(flights_model["Origin"].value_counts().reindex(list(ORIGIN_AIRPORTS)))
print(f"Overall DepDel15 rate: {flights_model['DepDel15'].mean():.3%}")
flights_model.head()